# Silver: Transformations

### 1.0: Configuration

In [0]:
%run ./00_utils

### 2.0: Silver Locations

In [ ]:
from pyspark.sql import functions as F

# QUALIFY keeps only the most recent daily snapshot of each location;
# the JSON stored as strings in bronze is parsed into typed columns.
df_locations_silver = spark.sql(f"""
    SELECT 
        id,
        name,
        locality,
        timezone,
        isMobile,
        isMonitor,
        CAST(get_json_object(coordinates, '$.latitude') AS DOUBLE) as latitude,
        CAST(get_json_object(coordinates, '$.longitude') AS DOUBLE) as longitude,
        get_json_object(country, '$.code') as country_code,
        get_json_object(country, '$.name') as country_name,
        get_json_object(owner, '$.name') as owner_name,
        get_json_object(provider, '$.name') as provider_name,
        CAST(get_json_object(datetimeLast, '$.utc') AS TIMESTAMP) as last_update_utc,
        ingested_at
    FROM {CATALOG}.{SCHEMA}.locations
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY ingested_at DESC) = 1
""")

# Data quality: drop locations with missing or impossible coordinates
df_locations_clean = df_locations_silver.filter(
    (F.col("latitude").isNotNull()) & 
    (F.col("longitude").isNotNull()) &
    (F.col("latitude").between(-90, 90)) &
    (F.col("longitude").between(-180, 180))
)

# Overwrite is safe here: every daily snapshot is complete, so a full rebuild
# is simpler than a MERGE and self-heals if a run was bad. Measurements are
# events, not snapshots; they are merged incrementally below.
df_locations_clean.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.locations")
print(f"Saved {df_locations_clean.count()} locations to silver (removed {df_locations_silver.count() - df_locations_clean.count()} invalid)")

### 3.0: Silver Sensors

In [ ]:
# Each location carries its sensors as a nested JSON array: parse the latest
# snapshot and explode it, one row per sensor with its location id.
df_sensors_silver = spark.sql(f"""
    WITH parsed_locations AS (
        SELECT 
            id as location_id,
            from_json(sensors, 'ARRAY<STRUCT<id: INT, name: STRING, parameter: STRUCT<id: INT, name: STRING, units: STRING, displayName: STRING>>>') as sensors_array,
            ingested_at
        FROM {CATALOG}.{SCHEMA}.locations
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY ingested_at DESC) = 1
    )
    SELECT DISTINCT
        sensor.id as sensor_id,
        sensor.name as sensor_name,
        sensor.parameter.id as parameter_id,
        location_id
    FROM parsed_locations
    LATERAL VIEW explode(sensors_array) AS sensor
    WHERE sensor.id IS NOT NULL
""")

df_sensors_silver.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.sensors")
print(f"Saved {df_sensors_silver.count()} sensors to silver")

### 4.0: Silver Parameters

In [ ]:
# Parameter metadata (pollutant name, units) lives in the same nested array.
# Same pattern as sensors: explode the latest snapshots, one row per parameter.
df_parameters_silver = spark.sql(f"""
    WITH parsed_locations AS (
        SELECT 
            from_json(sensors, 'ARRAY<STRUCT<id: INT, name: STRING, parameter: STRUCT<id: INT, name: STRING, units: STRING, displayName: STRING>>>') as sensors_array
        FROM {CATALOG}.{SCHEMA}.locations
        QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY ingested_at DESC) = 1
    )
    SELECT DISTINCT
        sensor.parameter.id as parameter_id,
        sensor.parameter.name as parameter_name,
        sensor.parameter.units as parameter_units,
        sensor.parameter.displayName as parameter_display_name
    FROM parsed_locations
    LATERAL VIEW explode(sensors_array) AS sensor
    WHERE sensor.parameter.id IS NOT NULL
""")

df_parameters_silver.write.mode("overwrite").saveAsTable(f"{CATALOG}.silver.parameters")
print(f"Saved {df_parameters_silver.count()} parameters to silver")

### 5.0: Silver Measurements

In [ ]:
# Incremental load: only bronze rows newer than the watermark, deduplicated on
# the business key (a sensor cannot produce two values at the same instant).
# The watermark is on ingestion time, so late readings are never dropped.

table_exists = spark.catalog.tableExists(f"{CATALOG}.silver.measurements")
watermark = (
    spark.sql(f"SELECT MAX(ingested_at) FROM {CATALOG}.silver.measurements").collect()[0][0]
    if table_exists
    else "1970-01-01"
)

# Validity depends on the parameter: negative values are instrument noise for
# concentrations but legitimate for temperature; the bounds also block -999 sentinels.
df_measurements_clean = spark.sql(f"""
    SELECT
        m.sensors_id,
        m.locations_id,
        m.value,
        CAST(get_json_object(m.datetime, '$.utc') AS TIMESTAMP) as datetime_utc,
        CAST(get_json_object(m.coordinates, '$.latitude') AS DOUBLE) as latitude,
        CAST(get_json_object(m.coordinates, '$.longitude') AS DOUBLE) as longitude,
        m.ingested_at
    FROM {CATALOG}.{SCHEMA}.measurements m
    LEFT JOIN {CATALOG}.silver.sensors s ON m.sensors_id = s.sensor_id
    LEFT JOIN {CATALOG}.silver.parameters p ON s.parameter_id = p.parameter_id
    WHERE m.ingested_at > '{watermark}'
      AND m.sensors_id IS NOT NULL
      AND m.locations_id IS NOT NULL
      AND m.value IS NOT NULL
      AND IF(p.parameter_name = 'temperature', m.value BETWEEN -60 AND 60, m.value >= 0)
      AND get_json_object(m.datetime, '$.utc') IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY m.sensors_id, datetime_utc ORDER BY m.ingested_at DESC) = 1
""")

if not table_exists:
    df_measurements_clean.write.saveAsTable(f"{CATALOG}.silver.measurements")
    print(f"Created silver.measurements with {df_measurements_clean.count()} rows")
else:
    df_measurements_clean.createOrReplaceTempView("new_measurements")
    spark.sql(f"""
        MERGE INTO {CATALOG}.silver.measurements AS target
        USING new_measurements AS source
        ON target.sensors_id = source.sensors_id
           AND target.datetime_utc = source.datetime_utc
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged {df_measurements_clean.count()} new rows into silver.measurements (watermark: {watermark})")

### 6.0: Verify Silver tables

In [ ]:
# Sanity check: row counts per silver table (visible in the job run output)
spark.sql(f"""
    SELECT 'locations' as table_name, COUNT(*) as rows FROM {CATALOG}.silver.locations
    UNION ALL
    SELECT 'sensors', COUNT(*) FROM {CATALOG}.silver.sensors
    UNION ALL
    SELECT 'parameters', COUNT(*) FROM {CATALOG}.silver.parameters
    UNION ALL
    SELECT 'measurements', COUNT(*) FROM {CATALOG}.silver.measurements
""").display()